# Task 7 : Inter-subject variation in the preterm cohort

Whole-WM group comparison, sex-stratified analysis, regional WM analysis, and correlation with cognitive outcomes.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
from utils import load_preterm_cohort, load_demographics
from analysis import (fit_cohort_wm, fit_regional, regional_group_stats,
                      ttest_summary, WM_REGIONS)
from plotting import (plot_group_boxplots, plot_sex_stratified,
                      plot_iq_scatter, plot_regional_grid, plot_ga_scatter)

## 7.1 : Load cohort with demographics

In [ ]:
preterm, cohort = load_preterm_cohort('../data', exclude_file='../data/preterm_exclusions.csv')
print(f"Cohort: n = {len(cohort)} (EP = {int(cohort['EP'].sum())}, FT = {int((~cohort['EP']).sum())})")
print("\nGestational age by group:")
print(cohort.groupby('EP')['gawks'].agg(['mean', 'std', 'min', 'max']).round(1))

## 7.2 : Per-subject WM fits

In [ ]:
df = fit_cohort_wm(preterm)
print(f"Fits: {len(df)} subjects")
print(df.head(10).round(3).to_string(index=False))

## 7.3 : Whole-WM group comparison (EP vs FT)

In [ ]:
for col in ['T2_WM', 'v_MWF']:
    ep = df.loc[df['EP'] == True, col].dropna().values
    ft = df.loc[df['EP'] == False, col].dropna().values
    r = ttest_summary(ep, ft)
    print(f"{col}: EP {r['mean_a']:.3f} vs FT {r['mean_b']:.3f}, "
          f"diff = {r['diff']:+.3f}, p = {r['p']:.4f}, d = {r['d']:.3f}")

plot_group_boxplots(df)

## 7.4 : Sex-stratified comparison

In [ ]:
df['id'] = pd.to_numeric(df['id'], errors='coerce').astype('Int64')
cohort['id'] = pd.to_numeric(cohort['id'], errors='coerce').astype('Int64')
df_merged = df.dropna(subset=['id']).merge(
    cohort[['id', 'male']].dropna(subset=['id']), on='id', how='inner')
plot_sex_stratified(df_merged)

## 7.5 : Correlation with cognitive outcome (FSIQ)

In [ ]:
merged = df.merge(cohort[['id', 'y19ps_fsiq_comp']], on='id')
plot_iq_scatter(merged)

## 7.6 : Regional WM analysis

In [ ]:
df_reg = fit_regional(preterm)
stats_df = regional_group_stats(df_reg)
print(f"Regional analysis: {len(stats_df)} tests")
print(stats_df.to_string(index=False))

plot_regional_grid(df_reg, stats_df, WM_REGIONS)

## 7.7 : WM parameters vs gestational age

In [ ]:
plot_ga_scatter(df)